# X-PrivScore — Experiment Runner (Colab)

**قبل التشغيل:** Runtime → Change runtime type → **T4 GPU**.

كل شيء يُحفظ على Google Drive (الكود، البيانات، الـ checkpoints، النتائج)،
فإذا انقطعت الجلسة أعيدي تشغيل الخلايا 1→3 ثم خلية التجربة نفسها — ستكمل من آخر checkpoint تلقائياً.

**المطلوب منك مرة واحدة فقط:**
1. ارفعي `xprivscore.zip` إلى جذر Google Drive.
2. جهزي `kaggle.json` (من kaggle.com → Settings → API → Create New Token) وارفعيه عند طلب الخلية 2.
3. في IEEE-CIS: لازم تكونين ضاغطة **Join Competition** على صفحة المسابقة في Kaggle أولاً وإلا التنزيل يفشل.

In [ ]:
# ── 1) Drive + code + deps ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
BASE = '/content/drive/MyDrive/xprivscore'
if not os.path.exists(BASE):
    with zipfile.ZipFile('/content/drive/MyDrive/xprivscore.zip') as z:
        z.extractall('/content/drive/MyDrive/')
%cd {BASE}
!pip -q install opacus shap kaggle
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# ── 2) Kaggle auth + download datasets (runs once; skipped if present) ──
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('ارفعي ملف kaggle.json:')
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

if not os.path.exists('data/creditcard.csv'):
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p data --unzip
if not os.path.exists('data/train_transaction.csv'):
    !kaggle competitions download -c ieee-fraud-detection -p data -f train_transaction.csv.zip
    !cd data && unzip -o train_transaction.csv.zip && rm -f train_transaction.csv.zip
if not os.path.exists('data/elliptic_txs_features.csv'):
    !kaggle datasets download -d ellipticco/elliptic-data-set -p data --unzip
    !find data -name 'elliptic_txs_*.csv' -exec mv {} data/ \\; 2>/dev/null
!ls -lh data/ | head

In [ ]:
# ── 3) Smoke test (أول مرة فقط؛ ~دقائق) ────────────────────────────
!python tests/smoke_test.py 2>&1 | grep -viE 'warning|warn|loss.backward' | tail -12

In [ ]:
# ── 4) Epsilon planning (بدون تدريب — ثوانٍ) ────────────────────────
# شغّليها وأرسلي المخرجات كاملة قبل التجارب الطويلة لاعتماد قيم sigma النهائية
for ds in ['ulb', 'ieee', 'elliptic']:
    for sg in ['1.2', '2.5', '4.0']:
        print(f'\n===== {ds} | sigma={sg} =====')
        !python run_experiment.py --plan --dataset {ds} --sigma {sg} | tail -3

## التجارب — بترتيب الأولوية
شغّلي خلية واحدة في كل جلسة. كل أمر يستكمل تلقائياً من checkpoint عند إعادة التشغيل.
الأزمنة تقديرية على T4؛ Elliptic الأسرع بفارق كبير فابدئي به للتأكد أن كل شيء سليم.

In [ ]:
# ── 5) الأولوية 1: Elliptic (الأسرع) — صف Table 3 ──────────────────
!python run_experiment.py --dataset elliptic --model tabtransformer --name t3_elliptic_tt
!python run_experiment.py --dataset elliptic --model mlp --name t3_elliptic_mlp

In [ ]:
# ── 6) الأولوية 2: ULB — صف Table 3 ─────────────────────────────────
!python run_experiment.py --dataset ulb --model tabtransformer --name t3_ulb_tt
!python run_experiment.py --dataset ulb --model mlp --name t3_ulb_mlp

In [ ]:
# ── 7) الأولوية 3: ULB — Table 4 (privacy–utility) و Table 5 (attacks) ──
!python run_experiment.py --dataset ulb --sigma 4.0 --name t4_sigma4
!python run_experiment.py --dataset ulb --sigma 2.5 --name t4_sigma25
!python run_experiment.py --dataset ulb --no-dp --name t4_nodp
for a in ['label_flip','scaling','collusion']:
    !python run_experiment.py --dataset ulb --attack {a} --name t5_{a}
    !python run_experiment.py --dataset ulb --attack {a} --no-dp --no-mpc --aggregation fedavg --name t5_{a}_flonly

In [ ]:
# ── 8) الأولوية 4: IEEE-CIS (الأثقل) — صف Table 3 ───────────────────
!python run_experiment.py --dataset ieee --model tabtransformer --name t3_ieee_tt
!python run_experiment.py --dataset ieee --model mlp --name t3_ieee_mlp

In [ ]:
# ── 9) الأولوية 5: Table 8 baselines (ULB) + AMR (Table 2) ──────────
!python run_experiment.py --dataset ulb --centralized --no-dp --name t8_central
!python run_experiment.py --dataset ulb --no-dp --no-mpc --aggregation fedavg --name t8_flonly
!python run_experiment.py --dataset ulb --no-mpc --name t8_fldp
!python run_experiment.py --dataset ulb --no-dp --name t8_flmpc
from amr import latency_benchmark, collusion_simulation, verify_correctness
print('cancellation err:', verify_correctness())
print(latency_benchmark(dim=18000, n=10, rounds=20))
print('collusion success (N=10,k=5,R=6):', collusion_simulation())

In [ ]:
# ── 10) تجميع النتائج لإرسالها ──────────────────────────────────────
!zip -rq /content/drive/MyDrive/xprivscore_results.zip results/
print('جاهز: MyDrive/xprivscore_results.zip — أرسليه في المحادثة')